# 06 · Rollout Strategies on Model Serving

**Goal:** replace a live model safely. Start a **canary** release (10% of traffic to the new model), then **promote** it,
and learn how a **rollback** works.

| Strategy | Traffic split | When to use it |
|---|---|---|
| **Canary** | New model gets a small share (e.g. 10%), raised step by step | The new model is promising but unproven |
| **A/B test** | Fixed split (e.g. 50/50) measured over time | Comparing two candidates on business results |
| **Blue-green** | 100% on old ("blue") while new ("green") is ready; switch at once | You need an instant switch and an instant way back |
| **Shadow** | New model gets a copy of requests; its answers are only logged | Zero risk to users |

**Time and cost:** creating an endpoint takes about 10 minutes and each traffic change about 5. The endpoint is deleted at
the end of the notebook (set `DELETE_ENDPOINT_AT_END = False` to keep it for the class).

In [ ]:
%run ./Includes/_config

In [ ]:
DELETE_ENDPOINT_AT_END = True

from datetime import timedelta
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput, TrafficConfig, Route
from mlflow.models.signature import infer_signature
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

w = WorkspaceClient()
MODEL = f"{CATALOG}.{SCHEMA}.diabetes_rollout_model"
ENDPOINT = f"sws-rollout-{slug}".replace("_", "-")[:60]

## 1. Register two model versions: the current model (v1) and a candidate (v2)

In [ ]:
pdf = spark.table(TABLE).sample(fraction=0.3, seed=3).toPandas()
X_tr, X_te, y_tr, y_te = train_test_split(pdf[FEATURES], pdf[LABEL], test_size=0.25, random_state=3)
versions = {}
for label, model in [("current", LogisticRegression(max_iter=1000)), ("candidate", GradientBoostingClassifier(random_state=3))]:
    model.fit(X_tr, y_tr)
    auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
    with mlflow.start_run(run_name=f"rollout_{label}"):
        mlflow.log_metric("test_auc", auc)
        info = mlflow.sklearn.log_model(model, "model", registered_model_name=MODEL,
                                        signature=infer_signature(X_tr.head(), model.predict(X_tr.head())),
                                        input_example=X_tr.head(3))
    versions[label] = str(info.registered_model_version)
    print(f"{label:9} {type(model).__name__:28} AUC {auc:.4f}  -> version {versions[label]}")

## 2. Canary: 90% to the current model, 10% to the candidate
Each **served entity** is one model version with a name. The **traffic config** decides how requests are split.

In [ ]:
def entity(name, version):
    return ServedEntityInput(name=name, entity_name=MODEL, entity_version=version,
                             workload_size="Small", scale_to_zero_enabled=True)

def traffic(current_pct):
    routes = [Route(served_model_name="current", traffic_percentage=current_pct)]
    if current_pct < 100:
        routes.append(Route(served_model_name="candidate", traffic_percentage=100 - current_pct))
    return TrafficConfig(routes=routes)

if any(e.name == ENDPOINT for e in w.serving_endpoints.list()):
    w.serving_endpoints.delete(ENDPOINT)          # start from a clean endpoint on re-runs

w.serving_endpoints.create_and_wait(
    name=ENDPOINT,
    config=EndpointCoreConfigInput(served_entities=[entity("current", versions["current"]), entity("candidate", versions["candidate"])],
                                   traffic_config=traffic(90)),
    timeout=timedelta(minutes=40))
print(f"Endpoint {ENDPOINT} is ready with a 90/10 canary split")

## 3. Send real requests
Apps call the endpoint with JSON rows of features and get predictions back. With a 90/10 split, roughly 1 request in
10 is answered by the candidate.

In [ ]:
response = w.serving_endpoints.query(name=ENDPOINT, dataframe_records=X_te.head(10).to_dict("records"))
print("Predictions:", response.predictions)
print("Actual:     ", y_te.head(10).astype(int).tolist())

## 4. Promote: send 100% of traffic to the candidate
After the canary looks healthy (errors, latency and business metrics), traffic is moved step by step. Here we jump to
100% to keep the demo short. The old version stays deployed until the next change, so going back is one call.

In [ ]:
w.serving_endpoints.update_config_and_wait(
    name=ENDPOINT,
    served_entities=[entity("current", versions["current"]), entity("candidate", versions["candidate"])],
    traffic_config=TrafficConfig(routes=[Route(served_model_name="current", traffic_percentage=0),
                                         Route(served_model_name="candidate", traffic_percentage=100)]),
    timeout=timedelta(minutes=40))
cfg = w.serving_endpoints.get(ENDPOINT).config
print("Traffic now:", [(r.served_model_name, r.traffic_percentage) for r in cfg.traffic_config.routes])

## 5. Rollback (reference)
If the candidate misbehaves, rollback is the same call with the percentages reversed:

```python
w.serving_endpoints.update_config_and_wait(
    name=ENDPOINT,
    served_entities=[entity("current", versions["current"]), entity("candidate", versions["candidate"])],
    traffic_config=TrafficConfig(routes=[Route(served_model_name="current", traffic_percentage=100),
                                         Route(served_model_name="candidate", traffic_percentage=0)]))
```

In the Serving UI, open the endpoint and look at **Served entities** and **Traffic** to see the split.

In [ ]:
if DELETE_ENDPOINT_AT_END:
    w.serving_endpoints.delete(ENDPOINT)
    print(f"Deleted {ENDPOINT} (it would keep its compute reserved otherwise)")
else:
    print(f"Kept {ENDPOINT} for the class. Delete it afterwards from Serving.")

## Key points
- One endpoint can serve **several model versions**; the traffic config decides who answers.
- **Canary → promote** reduces risk: only a small share of users meets an unproven model.
- **Rollback** is a configuration change, not a redeployment, so it is fast.
- Monitor each version's errors and results while traffic shifts (notebook 07 covers drift monitoring).